# 01 — Meet the RTS bus data

In this notebook, we will load a bus dataset, look at the table, and understand what its columns tell us. We are **not cleaning the data or building a model yet**.

## Before you begin

Put these two files **in the same folder**:

- `01_explore.ipynb` — this notebook
- `RTS_Route7_March2023.xlsx` — the Route 7 data

Open that folder in VS Code, open this notebook, and select your Python environment using **Select Kernel** at the top right.

**How to use this notebook:** read each explanation, click the code cell below it, and press **Shift + Enter**. Run cells in order. The result appears underneath the cell. A cell with `[*]` is still running; wait for it to finish.

You can write your answers by double-clicking the discussion cells and replacing the prompts.

## 1. Load the libraries

A **library** is a collection of tools we can use without writing everything ourselves.

- **pandas** helps us work with tables. We give it the short name `pd`.
- **matplotlib.pyplot** helps us make charts. We give it the short name `plt`.

`import` makes a library available in this notebook. It does not install it. If you see `ModuleNotFoundError`, follow the setup guide and check your selected kernel.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

No output is expected from the imports. If the cell finishes without an error, continue.

## 2. Load the Excel data

`pd.read_excel(...)` reads an Excel workbook. The filename goes inside quotation marks.

We store the result in a variable named **`df`**. It is a **dataframe**: a table with rows and columns, similar to an Excel sheet.

The Excel file must be in the same working folder as this notebook. `openpyxl` is the library pandas uses to read this `.xlsx` file; it is included in the setup instructions.

In [ ]:
df = pd.read_excel("RTS_Route7_March2023.xlsx", engine="openpyxl")

This loads the table into memory; it does not change the Excel file.

## 3. Show the dataframe

Type a variable name at the end of a code cell to display it. Large tables show only a preview. The full table is still stored in `df`.

You may need to scroll sideways to see all the columns.

In [ ]:
df

### Look closely

**One row represents a recorded stop visit by a bus.** It is not a complete journey. A bus can visit the same stop several times in a day.

The numbers at the far left, starting at 0, are the dataframe's **index** (row labels). They are not stop IDs.

To make the table easier to read, let's show only the first five rows. `head()` means “show the beginning.”

In [ ]:
df.head()

**Discuss together:** Can you find the bus number, stop name, and arrival time in the first row? What do you think the passenger columns mean?

## 4. How big is the table?

`df.shape` gives two numbers: **(number of rows, number of columns)**.

In [ ]:
df.shape

The Route 7 workbook should have **14,685 rows and 15 columns**. These are raw records, not 14,685 separate journeys or verified model examples.

## 5. What are the columns?

`df.columns` lists the column names. `.tolist()` presents them as a Python list.

Column names must be spelled exactly, including spaces and punctuation, when selecting them.

In [ ]:
df.columns.tolist()

### A guide to all 15 columns

| Column | What it tells us |
| --- | --- |
| `Route` | The recorded route label. |
| `Stop ID` | A label identifying the stop. |
| `Stop Name` | The name of the stop. |
| `Stop Seq ID` | The recorded position of the stop in a sequence. |
| `Day of Week` | The recorded weekday. |
| `Date` | The date of the record. |
| `Arrive` | The recorded arrival clock time. |
| `Passenger On` | Number of passengers boarding. |
| `Passenger Off` | Number of passengers leaving. |
| `Passenger Load` | Recorded number of passengers on the bus; we will check how it is defined. |
| `Passenger Miles` | A passenger-distance measure; its definition and units need checking. |
| `Interstop Distance` | A reported distance between stops; we must verify its units and segment alignment. |
| `Bus` | The vehicle identifier. This is not a journey identifier. |
| `Lat.` | Recorded latitude, a north–south coordinate. |
| `Long.` | Recorded longitude, an east–west coordinate. |

### Are these all model features?

Not automatically. These are the **raw columns**. A **feature** is an input we choose for a model. The **target** is the quantity we want the model to predict.

Our future target is **travel time from Stop A to the next Stop B, in seconds**. That column does not exist yet. Later, we will create it by subtracting two verified arrival timestamps.

Possible features include the stop pair, hour of day, and day of week. We cannot use the future arrival at B as an input: that would reveal the answer.

For now, let's learn to select the information we need.

## 6. Select one column

Square brackets select a column by name. The following cell shows the first five stop names.

- `df["Stop Name"]` selects the column.
- `.head()` limits the display to five rows.

In [ ]:
df["Stop Name"].head()

### Select several columns

For several columns, use a **list** of names inside the brackets. That is why you see two opening brackets: `[[`.

This smaller table is easier to follow than all 15 columns at once.

In [ ]:
df[["Bus", "Date", "Stop ID", "Stop Name", "Stop Seq ID", "Arrive"]].head(10)

**Your turn:** copy the cell above and change the list to display `Passenger On`, `Passenger Off`, and `Passenger Load`.

What do these columns suggest about passenger activity at the first few stops?

In [ ]:
# Your turn: select the three passenger columns and show their first five rows.


## 7. What kinds of values do the columns contain?

`df.info()` summarizes each column's name, number of non-missing values, and data type.

- `int64` usually means whole numbers.
- `float64` usually means numbers that can include decimal places.
- `datetime64` means dates/timestamps.
- `object` often means text or Python objects, including Excel clock times.

A column's storage type does not tell us how to use it. For example, a numeric stop ID is still a **label**, not a quantity to average.

In [ ]:
df.info()

## 8. Are any entries missing?

A missing entry is a blank or unknown value. `isna()` checks for missing entries. `.sum()` counts them in each column.

Do not replace missing values with zero automatically: “unknown” and “zero passengers” mean different things. Also, no blanks does not guarantee correct data.

In [ ]:
df.isna().sum()

**Discuss together:** Which columns have missing values? If there are none, what other problems could still exist?

## 9. How many buses, stops, and dates are represented?

`nunique()` counts distinct values. We are counting identifiers and recorded dates, not journeys.

In [ ]:
print("Different buses:", df["Bus"].nunique())
print("Different stops:", df["Stop ID"].nunique())
print("Different dates:", df["Date"].nunique())

### How many recorded rows are there for each date?

`value_counts()` counts how often each value appears. `sort_index()` puts the date labels in order.

Dates with fewer rows may have less coverage. These counts alone do not tell us how many buses operated.

In [ ]:
rows_per_date = df["Date"].value_counts().sort_index()
rows_per_date

## 10. Make a first chart

We use the counts from the previous cell to draw a bar chart. Each bar represents the number of recorded stop visits on one observed date.

Missing dates are not shown as zero-service days. These records are an incomplete sample of March 2023.

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(rows_per_date.index.astype(str), rows_per_date.values)
plt.title("Route 7: recorded stop visits by date")
plt.xlabel("Recorded date")
plt.ylabel("Number of rows")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

**Discuss together:** Are the bars similar heights? What might explain the differences? What information would we need before claiming service changed?

## 11. Look for repeated event details

Later we need reliable stop-to-stop examples. First, let's ask whether some rows share the same **date, bus, stop, and arrival time**.

`duplicated(...)` marks later occurrences of the same combination. Adding `.sum()` counts the extra occurrences. This does **not** mean the full rows are identical.

In [ ]:
event_columns = ["Date", "Bus", "Stop ID", "Arrive"]
df.duplicated(subset=event_columns).sum()

Let's look at a few of these rows. `keep=False` marks **all members** of a repeated group, so we can compare them.

The first line creates a True/False selection. The second line shows rows where the selection is True. We are inspecting them, not deleting them.

In [ ]:
repeated_rows = df.duplicated(subset=event_columns, keep=False)
df.loc[repeated_rows].head(10)

**Discuss together:** Do the passenger or distance values differ? Should we remove these rows immediately, or investigate first? Notebook 2 will help us make that decision.

## 12. Write your group's observations

Edit this text cell and answer in your own words:

1. **One row in our table represents:**
2. **The table has this many rows and columns:**
3. **Three columns I can explain are:**
4. **A feature we might use later is:**
5. **The target we want to create is:**
6. **One thing I noticed in the chart or repeated rows is:**
7. **Three questions to investigate before cleaning are:**

### Before finishing

- Save your notebook with **Ctrl + S** (Windows) or **Cmd + S** (Mac).
- Make sure you can explain the code, not only run it.
- Keep the Excel file unchanged.

**Next notebook:** clean records carefully and verify which stop visits belong together before calculating travel times.

### If the Excel file cannot be found

Check that its name is exactly `RTS_Route7_March2023.xlsx`. Open the folder containing **both the notebook and the workbook** in VS Code. Restart the kernel and run from the top. If it still fails, ask your instructor to check the notebook's working folder.

### Data credit

Richard Borzi, University of Florida; data supplied by Gainesville RTS. [Route 7 dataset](https://zenodo.org/records/10034933), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). These are historical, incomplete March 2023 records.